# 02 Generate Synthetic Data And Classifier Experiments

Notebook-native synthetic image generation followed by classifier experiments. Run this notebook top to bottom after `01_data_setup_and_train_gan.ipynb` when you want to generate a synthetic pool and immediately train or compare classifier setups against it.


In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import os
import random
import sys
import time

import torch
import torch.nn as nn
from PIL import Image
from IPython.display import display
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import save_image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from models.classifier import FruitCNN
from models.gan import Generator

ROOT


In [ ]:
CLASS_NAMES = ["apple", "banana", "orange"]


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def clear_synth_dir(out_root, class_names=None):
    out_root = Path(out_root)
    class_names = class_names or CLASS_NAMES
    for class_name in class_names:
        class_dir = out_root / class_name
        if not class_dir.exists():
            continue
        for image_path in class_dir.glob("*.png"):
            image_path.unlink()


def generate_synth_pool_notebook(cfg, ckpt, n_per_class, out_dir, batch_size=64, seed=None, class_names=None):
    ckpt = Path(ckpt)
    out_root = Path(out_dir)
    class_names = class_names or CLASS_NAMES
    seed = cfg.seed if seed is None else seed
    device = torch.device(cfg.resolve_device())

    G = Generator(z_dim=cfg.z_dim, num_classes=len(class_names)).to(device)
    ckpt_state = load_checkpoint(ckpt, device)
    G.load_state_dict(ckpt_state["G"])
    G.eval()

    torch.manual_seed(seed)
    start_time = time.time()
    counts = {}
    clear_synth_dir(out_root, class_names)

    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = out_root / cls_name
        cls_dir.mkdir(parents=True, exist_ok=True)

        generated = 0
        while generated < n_per_class:
            bs = min(batch_size, n_per_class - generated)
            z = torch.randn(bs, cfg.z_dim, device=device)
            y = torch.full((bs,), cls_idx, dtype=torch.long, device=device)
            with torch.no_grad():
                imgs = G(z, y)

            for i in range(bs):
                save_image(
                    imgs[i],
                    cls_dir / f"{cls_name}_synth_{generated + i:05d}.png",
                    normalize=True,
                    value_range=(-1, 1),
                )
            generated += bs
        counts[cls_name] = generated

    summary = {
        "checkpoint": str(ckpt),
        "out_dir": str(out_root),
        "n_per_class": n_per_class,
        "counts": counts,
        "seed": seed,
        "generate_time_sec": round(time.time() - start_time, 1),
    }
    with open(out_root / "generation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


In [ ]:
def get_transform(img_size, train=True, augmentation_policy="none"):
    tf_list = [transforms.Resize((img_size, img_size))]
    if train and augmentation_policy == "classical":
        tf_list.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ])
    tf_list.extend([
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ])
    return transforms.Compose(tf_list)


def scenario_augmentation_policy(scenario):
    return "classical" if scenario == "real_aug" else "none"


def scenario_time_breakdown(scenario, generation_summary=None):
    if not generation_summary or scenario not in {"synth", "both"}:
        return {}
    return {
        "synth_generation_time_sec": round(float(generation_summary.get("generate_time_sec", 0.0)), 1)
    }


def subsample_dataset(dataset, n_per_class, seed=42):
    rng = random.Random(seed)
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices[label].append(idx)
    selected = []
    for label, indices in sorted(class_indices.items()):
        rng.shuffle(indices)
        selected.extend(indices[:n_per_class])
    return Subset(dataset, selected)


def build_dataset(cfg, scenario, n_per_class, synth_dir="data_synth", real_train_root=None, test_root=None):
    augmentation_policy = scenario_augmentation_policy(scenario)
    tf_train = get_transform(cfg.img_size, train=True, augmentation_policy=augmentation_policy)
    tf_test = get_transform(cfg.img_size, train=False, augmentation_policy="none")

    real_train_root = Path(real_train_root) if real_train_root else cfg.data_root / "train"
    test_root = Path(test_root) if test_root else cfg.data_root / "test"
    synth_dir = Path(synth_dir)

    real_train = datasets.ImageFolder(str(real_train_root), transform=tf_train)
    test_ds = datasets.ImageFolder(str(test_root), transform=tf_test)

    if scenario in {"real", "real_aug"}:
        train_ds = subsample_dataset(real_train, n_per_class, cfg.seed) if n_per_class else real_train
    elif scenario == "synth":
        if not synth_dir.exists():
            raise FileNotFoundError(f"Synthetic directory not found: {synth_dir}")
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        train_ds = subsample_dataset(synth_train, n_per_class, cfg.seed) if n_per_class else synth_train
    elif scenario == "both":
        if not synth_dir.exists():
            raise FileNotFoundError(f"Synthetic directory not found: {synth_dir}")
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        if n_per_class:
            real_sub = subsample_dataset(real_train, n_per_class, cfg.seed)
            synth_sub = subsample_dataset(synth_train, n_per_class, cfg.seed)
            train_ds = ConcatDataset([real_sub, synth_sub])
        else:
            train_ds = ConcatDataset([real_train, synth_train])
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    return train_ds, test_ds, real_train.classes, augmentation_policy


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    return all_preds, all_labels


def run_classifier_notebook(cfg, scenario, n_per_class, synth_dir, out_dir, real_train_root=None, test_root=None, time_breakdown=None, extra_metadata=None):
    from sklearn.metrics import classification_report

    device = torch.device(cfg.resolve_device())
    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)

    train_ds, test_ds, class_names, augmentation_policy = build_dataset(
        cfg,
        scenario,
        n_per_class,
        synth_dir=synth_dir,
        real_train_root=real_train_root,
        test_root=test_root,
    )

    loader_kwargs = cfg.loader_options()
    train_loader = DataLoader(train_ds, batch_size=cfg.clf_batch, shuffle=True, drop_last=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=cfg.clf_batch, shuffle=False, **loader_kwargs)

    model = FruitCNN(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.clf_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.clf_epochs)

    print(f"[{scenario}] n_per_class={n_per_class or 'all'} train_size={len(train_ds)} device={device}")
    t0 = time.time()
    for epoch in range(1, cfg.clf_epochs + 1):
        loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
        if epoch % 10 == 0 or epoch == cfg.clf_epochs:
            print(f"  Epoch {epoch:03d}  loss={loss:.4f}  train_acc={acc:.4f}")
    train_time = time.time() - t0

    preds, labels = evaluate(model, test_loader, device)
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    test_acc = report["accuracy"]

    time_breakdown = time_breakdown or {}
    gan_train_time = round(float(time_breakdown.get("gan_train_time_sec", 0.0)), 1)
    synth_generation_time = round(float(time_breakdown.get("synth_generation_time_sec", 0.0)), 1)
    pipeline_time = round(train_time + gan_train_time + synth_generation_time, 1)

    result = {
        "scenario": scenario,
        "augmentation_policy": augmentation_policy,
        "n_per_class": n_per_class or "all",
        "train_size": len(train_ds),
        "test_accuracy": round(test_acc, 4),
        "train_time_sec": round(train_time, 1),
        "classifier_train_time_sec": round(train_time, 1),
        "gan_train_time_sec": gan_train_time,
        "synth_generation_time_sec": synth_generation_time,
        "pipeline_time_sec": pipeline_time,
        "real_train_root": str(real_train_root or (cfg.data_root / "train")),
        "test_root": str(test_root or (cfg.data_root / "test")),
        "synth_dir": str(Path(synth_dir)),
        "per_class": {
            name: {
                "precision": round(report[name]["precision"], 4),
                "recall": round(report[name]["recall"], 4),
                "f1": round(report[name]["f1-score"], 4),
            }
            for name in class_names
        },
    }
    if extra_metadata:
        result.update(extra_metadata)

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    tag = f"{scenario}_n{n_per_class}" if n_per_class else f"{scenario}_all"
    with open(out_path / f"result_{tag}.json", "w") as f:
        json.dump(result, f, indent=2)
    return result


def run_grid_notebook(cfg, sizes, scenarios, synth_dir, out_dir, real_train_root=None, test_root=None, generation_summary=None):
    all_results = []
    for n in sizes:
        for scenario in scenarios:
            result = run_classifier_notebook(
                cfg,
                scenario,
                n,
                synth_dir,
                out_dir,
                real_train_root=real_train_root,
                test_root=test_root,
                time_breakdown=scenario_time_breakdown(scenario, generation_summary),
            )
            all_results.append(result)
    out_path = Path(out_dir)
    with open(out_path / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    return all_results


In [ ]:
# Edit these values before running.
cfg = Config()
CKPT = ROOT / "runs" / "gan" / "checkpoints" / "best_fid.pt"
SYNTH_DIR = ROOT / "data_synth"
SYNTH_N_PER_CLASS = 1300
SYNTH_BATCH_SIZE = 64
SEED = cfg.seed

SCENARIO = "synth"
CLF_N_PER_CLASS = 400
REAL_TRAIN_ROOT = ROOT / "data_final" / "train"
TEST_ROOT = ROOT / "data_final" / "test"
CLF_OUT_DIR = ROOT / "runs" / "clf"

cfg, CKPT, SYNTH_DIR, SCENARIO, CLF_N_PER_CLASS


In [ ]:
generation_summary = generate_synth_pool_notebook(
    cfg=cfg,
    ckpt=CKPT,
    n_per_class=SYNTH_N_PER_CLASS,
    out_dir=SYNTH_DIR,
    batch_size=SYNTH_BATCH_SIZE,
    seed=SEED,
)
generation_summary


In [ ]:
for class_name in CLASS_NAMES:
    sample_path = sorted((SYNTH_DIR / class_name).glob("*.png"))[0]
    print(class_name, sample_path.name)
    display(Image.open(sample_path))


In [ ]:
single_result = run_classifier_notebook(
    cfg,
    SCENARIO,
    CLF_N_PER_CLASS,
    SYNTH_DIR,
    CLF_OUT_DIR,
    real_train_root=REAL_TRAIN_ROOT,
    test_root=TEST_ROOT,
    time_breakdown=scenario_time_breakdown(SCENARIO, generation_summary),
)
single_result


In [ ]:
# Optional small grid.
GRID_SIZES = [100, 200]
GRID_SCENARIOS = ["real", "synth", "both", "real_aug"]
# grid_results = run_grid_notebook(
#     cfg,
#     GRID_SIZES,
#     GRID_SCENARIOS,
#     SYNTH_DIR,
#     CLF_OUT_DIR,
#     real_train_root=REAL_TRAIN_ROOT,
#     test_root=TEST_ROOT,
#     generation_summary=generation_summary,
# )
